# Pangeo Climate Analysis Platform - Quick Start Guide

This notebook demonstrates the core functionality of the cloud-native climate analysis platform:

1. Cloud data access (S3/Zarr)
2. CMIP6 catalog search and loading
3. Dask Gateway for parallel computing
4. Climate analysis workflows

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client

from climate_analysis import (
    CloudDataStore,
    CMIP6Catalog,
    DaskGatewayCluster,
    TrendAnalysis,
    ClimateVisualizer
)

%matplotlib inline

## 1. Cloud Data Access (S3/Zarr)

In [ ]:
# Connect to S3 bucket (anonymous access for public data)
store = CloudDataStore(
    bucket="pangeo-data", 
    endpoint_url="https://s3.amazonaws.com",
    anon=True
)

In [ ]:
# List available objects in a prefix
try:
    objects = store.list_objects("cmip6")
    print(f"Found {len(objects)} objects in cmip6 prefix")
    print("First 5 objects:")
    for obj in objects[:5]:
        print(f"  - {obj}")
except Exception as e:
    print(f"Note: S3 access example skipped: {e}")
    print("We'll use simulated data for this demo")

## 2. CMIP6 Catalog Search

In [ ]:
# Load the Pangeo CMIP6 catalog
try:
    cat = CMIP6Catalog()
    print(f"Catalog loaded with {len(cat.catalog.df)} entries")
except Exception as e:
    print(f"Note: CMIP6 catalog skipped: {e}")

In [ ]:
# Search for historical temperature data from CESM2
if 'cat' in locals():
    subset = cat.search(
        experiment_id="historical",
        variable_id="tas",
        source_id="CESM2",
        member_id="r1i1p1f1"
    )
    print(f"Found {len(subset.df)} matching datasets")
    print(subset.df[['source_id', 'experiment_id', 'variable_id', 'member_id']])

## 3. Dask Gateway for Parallel Computing

In [ ]:
# Connect to Dask Gateway and create a cluster
try:
    cluster_manager = DaskGatewayCluster()
    
    # List existing clusters
    existing_clusters = cluster_manager.list_clusters()
    print(f"Existing clusters: {existing_clusters}")
    
    # Create a new cluster with custom resources
    cluster = cluster_manager.new_cluster(
        worker_cores=2,
        worker_memory="8G",
        worker_count=4,
        autoscale=True
    )
    
    # Get the client
    client = cluster_manager.get_client()
    print(f"\nDask cluster created!")
    print(f"Cluster name: {cluster.name}")
    print(f"Dashboard link: {client.dashboard_link}")
    
except Exception as e:
    print(f"Note: Dask Gateway example skipped (running in local mode)")
    print(f"Reason: {e}")
    
    # Create a local Dask client for demonstration
    client = Client(n_workers=2, threads_per_worker=2)
    print(f"Local Dask client created: {client}")

## 4. Load and Analyze Climate Data

In [ ]:
# Create sample data for demonstration
# (In production, this would be loaded from S3/Zarr)

# Create coordinate arrays
lats = np.linspace(-89.5, 89.5, 90)
lons = np.linspace(0.5, 359.5, 180)
times = pd.date_range(start="2000-01-01", periods=240, freq="MS")

# Create sample temperature data with trend
lat_grid, lon_grid, time_grid = np.meshgrid(lats, lons, np.arange(len(times)), indexing="ij")

# Base climate pattern
base_temp = 15 * np.cos(np.deg2rad(lat_grid)) - 5 * np.cos(2 * np.deg2rad(lat_grid))

# Add trend
trend = 0.01 * time_grid  # ~0.12 K per year

# Add seasonal cycle
seasonal = 10 * np.cos(2 * np.pi * (time_grid % 12) / 12)

# Add noise
noise = np.random.randn(*lat_grid.shape) * 0.5

# Combine components
temp_data = base_temp + trend + seasonal + noise

In [ ]:
# Create xarray Dataset
ds = xr.Dataset(
    {
        'tas': (["lat", "lon", "time"], temp_data),
    },
    coords={
        "lat": lats,
        "lon": lons,
        "time": times,
    },
)

print("Dataset created:")
print(ds)
print(f"\nData size: {ds.nbytes / 1e6:.2f} MB")

## 5. Trend Analysis

In [ ]:
# Initialize trend analysis
trend_analysis = TrendAnalysis(ds.tas)

# Compute trends with statistical significance
print("Computing trends...")
trend, p_value = trend_analysis.linear_trend()
print("Trend computation complete!")

In [ ]:
# Print summary statistics
print("Trend Summary (K/20 years):")
print(f"  Mean trend: {trend.mean().values:.4f}")
print(f"  Max trend: {trend.max().values:.4f}")
print(f"  Min trend: {trend.min().values:.4f}")

sig_percent = (p_value < 0.05).mean().values * 100
print(f"\nSignificant at p<0.05: {sig_percent:.1f}%")

## 6. Visualization

In [ ]:
# Initialize visualizer
viz = ClimateVisualizer(figsize=(14, 8))

In [ ]:
# Plot temperature trend with significance markers
print("Plotting temperature trend...")
fig_trend = viz.plot_trend_with_significance(
    trend,
    p_value,
    alpha=0.05,
    title="Temperature Trend 2000-2019 (dots indicate p<0.05 significance)",
)
plt.show()

In [ ]:
# Plot spatial mean time series
print("Plotting spatial mean time series...")
spatial_mean = ds.tas.mean(dim=["lat", "lon"])

fig_ts = viz.plot_time_series(
    spatial_mean,
    title="Global Mean Temperature (2000-2019)",
    ylabel="Temperature (K)",
)
plt.show()

## 7. Advanced: Compute EOFs

In [ ]:
from climate_analysis import EOFAnalysis

# Compute anomalies
tas_anomaly = ds.tas - ds.tas.mean(dim="time")

# EOF analysis
print("Performing EOF analysis...")
eof_analysis = EOFAnalysis(tas_anomaly)
eofs, pcs, eigenvalues = eof_analysis.fit(n_modes=4, apply_weights=True)
print("EOF analysis complete!")

In [ ]:
# Plot EOF modes
print("Plotting EOF modes...")
explained_variance = eof_analysis.get_explained_variance_ratio()
fig_eof = viz.plot_eof_modes(eofs, explained_variance, n_modes=4)
plt.show()

In [ ]:
# Plot PC time series
print("Plotting principal components...")
fig_pcs = viz.plot_pcs(pcs, explained_variance, n_modes=4)
plt.show()

## 8. Save Results to Cloud Storage

In [ ]:
# Create output dataset
output_ds = xr.Dataset({
    "trend": trend,
    "p_value": p_value,
    "eofs": eofs,
    "pcs": pcs,
    "explained_variance": ("mode", explained_variance),
})

print("Output dataset:")
print(output_ds)

In [ ]:
# Save to local Zarr store (in production, save to S3)
output_path = "/tmp/climate_analysis_results.zarr"
print(f"Saving results to {output_path}...")
output_ds.to_zarr(output_path, mode="w", consolidated=True)
print("Results saved successfully!")

## 9. Cleanup

In [ ]:
# Shutdown Dask cluster if using Dask Gateway
if 'cluster_manager' in locals():
    try:
        cluster_manager.stop_cluster()
        print("Dask cluster shut down")
    except Exception as e:
        print(f"Cluster cleanup note: {e}")

## Summary

In this notebook, we demonstrated:

1. **Cloud Data Access**: Connecting to S3/Zarr cloud storage
2. **CMIP6 Catalog**: Searching and filtering climate datasets
3. **Dask Parallelism**: Using Dask Gateway for scalable computing
4. **Trend Analysis**: Computing temperature trends with statistical significance
5. **EOF Analysis**: Empirical Orthogonal Function decomposition
6. **Visualization**: Creating publication-quality plots
7. **Cloud Storage**: Saving results to cloud-native formats

This platform enables scalable, reproducible climate analysis on petabyte-scale datasets using cloud-native technologies!